# Bangla Cyberbullying Detection - Model Compression Framework

This notebook provides a complete pipeline for compressing cyberbullying detection models using:
- **Knowledge Distillation (KD)**: Transfer knowledge from large teacher to small student
- **Pruning**: Remove unnecessary weights (magnitude, gradual, wanda)
- **Quantization**: Reduce precision (FP16, INT8, INT4)

## Models Used
- **Teacher**: `Saif-Siddique/bangla-cyberbully-xlm-roberta-base` (finetuned)
- **Student**: `neuropark/sahajBERT` (pretrained, not finetuned)

## Labels
Multi-label classification with 5 categories:
- `bully`: General cyberbullying
- `sexual`: Sexual harassment
- `religious`: Religious hate
- `threat`: Threats/violence
- `spam`: Spam content

---
## Cell 1: Environment Setup

In [ ]:
# Install required packages
!pip install transformers datasets accelerate bitsandbytes -q
!pip install scikit-learn pandas numpy tqdm -q

import os
import sys
import torch
import pandas as pd
import numpy as np
import json
from datetime import datetime

print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Cell 2: Mount Dataset and Navigate to Working Directory

In [ ]:
# Option A: If you uploaded the codebase as a Kaggle dataset
# Replace 'your-dataset-name' with your actual dataset name
CODEBASE_DATASET = "your-username/cyberbully-compression-framework"
DATA_DATASET = "your-username/bangla-cyberbully-data"

# Copy codebase to working directory
!cp -r /kaggle/input/cyberbully-compression-framework/* /kaggle/working/
%cd /kaggle/working

# Create data directory and copy dataset
!mkdir -p ./data
!cp /kaggle/input/bangla-cyberbully-data/*.csv ./data/

# List files to verify
print("\n" + "=" * 60)
print("FILES IN WORKING DIRECTORY")
print("=" * 60)
!ls -la

print("\n" + "=" * 60)
print("FILES IN DATA DIRECTORY")
print("=" * 60)
!ls -la ./data/

---
## Cell 3: Verify Dataset Structure

In [ ]:
# Load and inspect dataset
DATASET_PATH = "./data/1_Multilablel_Cyberbully_Data.csv"

df = pd.read_csv(DATASET_PATH)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)
label_columns = ['bully', 'sexual', 'religious', 'threat', 'spam']
for col in label_columns:
    if col in df.columns:
        positives = df[col].sum()
        percentage = df[col].mean() * 100
        print(f"  {col:12s}: {positives:5d} positives ({percentage:5.1f}%)")

print("\n" + "=" * 60)
print("SAMPLE DATA")
print("=" * 60)
print(df.head(3).to_string())

---
## Cell 4: Inspect Teacher and Student Models

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoConfig

# Model paths
TEACHER_MODEL = "Saif-Siddique/bangla-cyberbully-xlm-roberta-base"
STUDENT_MODEL = "neuropark/sahajBERT"

print("=" * 60)
print("TEACHER MODEL")
print("=" * 60)
print(f"Path: {TEACHER_MODEL}")
try:
    teacher_config = AutoConfig.from_pretrained(TEACHER_MODEL)
    print(f"  Architecture: {teacher_config.model_type}")
    print(f"  Hidden Size: {teacher_config.hidden_size}")
    print(f"  Num Layers: {teacher_config.num_hidden_layers}")
    print(f"  Num Attention Heads: {teacher_config.num_attention_heads}")
    print(f"  Vocab Size: {teacher_config.vocab_size}")
except Exception as e:
    print(f"  Error loading config: {e}")

print("\n" + "=" * 60)
print("STUDENT MODEL")
print("=" * 60)
print(f"Path: {STUDENT_MODEL}")
try:
    student_config = AutoConfig.from_pretrained(STUDENT_MODEL)
    print(f"  Architecture: {student_config.model_type}")
    print(f"  Hidden Size: {student_config.hidden_size}")
    print(f"  Num Layers: {student_config.num_hidden_layers}")
    print(f"  Num Attention Heads: {student_config.num_attention_heads}")
    print(f"  Vocab Size: {student_config.vocab_size}")
except Exception as e:
    print(f"  Error loading config: {e}")

# Load models to compare sizes
print("\n" + "=" * 60)
print("SIZE COMPARISON")
print("=" * 60)
try:
    teacher_model = AutoModel.from_pretrained(TEACHER_MODEL)
    student_model = AutoModel.from_pretrained(STUDENT_MODEL)
    
    teacher_params = sum(p.numel() for p in teacher_model.parameters())
    student_params = sum(p.numel() for p in student_model.parameters())
    
    print(f"  Teacher: {teacher_params/1e6:.1f}M parameters")
    print(f"  Student: {student_params/1e6:.1f}M parameters")
    print(f"  Reduction: {(1 - student_params/teacher_params)*100:.1f}%")
    
    # Clean up memory
    del teacher_model, student_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
except Exception as e:
    print(f"  Error: {e}")

---
## Cell 5: Configuration Variables

Set your experiment configuration here. All scenarios will use these values.

In [ ]:
# =============================================================================
# CONFIGURATION - MODIFY THESE VALUES
# =============================================================================

# Paths
DATASET_PATH = "./data/1_Multilablel_Cyberbully_Data.csv"
RESULTS_DIR = "./results"

# Models
TEACHER_CHECKPOINT = "Saif-Siddique/bangla-cyberbully-xlm-roberta-base"
STUDENT_PATH = "neuropark/sahajBERT"

# Author
AUTHOR_NAME = "Saif-Siddique"

# Knowledge Distillation
KD_EPOCHS = 5
KD_LEARNING_RATE = 2e-5
KD_ALPHA = 0.7  # Weight for soft loss (1-alpha for hard loss)
KD_TEMPERATURE = 4.0
KD_BATCH_SIZE = 16

# Pruning
PRUNING_METHOD = "magnitude"  # magnitude, gradual, wanda
TARGET_SPARSITY = 0.3  # 30% of weights removed
PRUNE_EPOCHS = 3

# Quantization
QUANTIZATION_TYPE = "int8"  # fp16, int8, int4

# Development mode (set to 1.0 for full training)
DATA_FRACTION = 1.0  # Use 0.1 for quick tests

# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration set successfully!")
print(f"Results will be saved to: {RESULTS_DIR}")

---
## Cell 6: Quick Test - Verify Codebase Works

Run a quick test with 5% of data and 1 epoch to verify everything works.

In [ ]:
%%time
# Quick test with minimal data
!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --data_fraction 0.05 \
    --kd_epochs 1 \
    --output_dir {RESULTS_DIR}/test_run

print("\n" + "=" * 60)
print("TEST COMPLETE")
print("=" * 60)
print("If you see no errors above, the codebase is working correctly!")

---
# SCENARIO 1: BASELINE

**Purpose:** Establish performance baseline of teacher model (no compression)

**What happens:**
- Loads teacher model from HuggingFace
- Evaluates on test set
- Records metrics (F1, latency, model size)

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 1: BASELINE")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline baseline \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario1_baseline

---
# SCENARIO 2: KD ONLY (Knowledge Distillation)

**Purpose:** Transfer knowledge from teacher to smaller student model

**What happens:**
- Teacher generates soft labels (probabilities)
- Student learns from both soft labels and ground truth
- Result: Smaller, faster model with similar accuracy

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 2: KD ONLY")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --kd_learning_rate {KD_LEARNING_RATE} \
    --kd_alpha {KD_ALPHA} \
    --kd_temperature {KD_TEMPERATURE} \
    --kd_batch_size {KD_BATCH_SIZE} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario2_kd_only

---
# SCENARIO 3: PRUNE ONLY

**Purpose:** Make teacher model sparser by removing unimportant weights

**What happens:**
- Identifies least important weights (by magnitude)
- Sets them to zero
- Fine-tunes to recover accuracy
- Result: Sparser model, potentially faster inference

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 3: PRUNE ONLY")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline prune_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --prune_epochs {PRUNE_EPOCHS} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario3_prune_only

---
# SCENARIO 4: QUANT ONLY (Quantization)

**Purpose:** Reduce precision of teacher model weights

**What happens:**
- Converts FP32 weights to INT8 (or FP16/INT4)
- Result: ~4x smaller model (for INT8), faster inference

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 4: QUANT ONLY")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline quant_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --quantization_type {QUANTIZATION_TYPE} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario4_quant_only

---
# SCENARIO 5: KD + PRUNE

**Purpose:** Two-stage compression - distill then prune

**What happens:**
1. Knowledge distillation to student
2. Prune the distilled student
3. Fine-tune to recover accuracy

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 5: KD + PRUNE")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --kd_alpha {KD_ALPHA} \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --prune_epochs {PRUNE_EPOCHS} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario5_kd_prune

---
# SCENARIO 6: KD + QUANT

**Purpose:** Distill then quantize

**What happens:**
1. Knowledge distillation to student
2. Quantize the distilled student to INT8

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 6: KD + QUANT")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --quantization_type {QUANTIZATION_TYPE} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario6_kd_quant

---
# SCENARIO 7: PRUNE + QUANT

**Purpose:** Prune teacher then quantize (no KD)

**What happens:**
1. Prune teacher model
2. Quantize the pruned teacher

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 7: PRUNE + QUANT")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline prune_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --prune_epochs {PRUNE_EPOCHS} \
    --quantization_type {QUANTIZATION_TYPE} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario7_prune_quant

---
# SCENARIO 8: FULL PIPELINE (KD + PRUNE + QUANT)

**Purpose:** Maximum compression using all three techniques

**What happens:**
1. Knowledge distillation to student
2. Prune the distilled student
3. Quantize the pruned student
4. Result: Smallest, fastest model

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 8: FULL PIPELINE (KD + PRUNE + QUANT)")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --kd_alpha {KD_ALPHA} \
    --kd_temperature {KD_TEMPERATURE} \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --prune_epochs {PRUNE_EPOCHS} \
    --quantization_type {QUANTIZATION_TYPE} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario8_full_pipeline

---
# SCENARIO 9: PRUNING METHOD COMPARISON

Compare different pruning algorithms:
- **Magnitude**: Remove smallest absolute weights
- **Gradual**: Prune incrementally during training
- **Wanda**: Activation-aware pruning (state-of-the-art)

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 9A: MAGNITUDE PRUNING")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --pruning_method magnitude \
    --target_sparsity 0.5 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario9a_magnitude

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 9B: GRADUAL PRUNING")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --pruning_method gradual \
    --target_sparsity 0.5 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario9b_gradual

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 9C: WANDA PRUNING")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --pruning_method wanda \
    --target_sparsity 0.5 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario9c_wanda

---
# SCENARIO 10: QUANTIZATION TYPE COMPARISON

Compare different quantization levels:
- **FP16**: Half precision (minimal quality loss)
- **INT8**: 8-bit integer (good balance)
- **INT4**: 4-bit integer (maximum compression)

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 10A: FP16 QUANTIZATION")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --quantization_type fp16 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario10a_fp16

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 10B: INT8 QUANTIZATION")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --quantization_type int8 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario10b_int8

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 10C: INT4 QUANTIZATION")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --quantization_type int4 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario10c_int4

---
# SCENARIO 11: SPARSITY LEVEL COMPARISON

Compare different pruning levels (20%, 30%, 50%, 70%)

In [ ]:
%%time
sparsity_levels = [0.2, 0.3, 0.5, 0.7]

for sparsity in sparsity_levels:
    print("\n" + "=" * 60)
    print(f"SCENARIO 11: SPARSITY {int(sparsity*100)}%")
    print("=" * 60)
    
    !python main.py \
        --dataset_path {DATASET_PATH} \
        --author_name "{AUTHOR_NAME}" \
        --pipeline kd_prune \
        --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
        --student_path "{STUDENT_PATH}" \
        --kd_epochs {KD_EPOCHS} \
        --pruning_method magnitude \
        --target_sparsity {sparsity} \
        --data_fraction {DATA_FRACTION} \
        --output_dir {RESULTS_DIR}/scenario11_sparsity_{int(sparsity*100)}

---
# SCENARIO 12: K-FOLD CROSS VALIDATION

**Purpose:** Robust evaluation across all K folds

**What happens:**
- Runs the pipeline on all 5 folds
- Aggregates results (mean +/- std)
- More reliable performance estimate

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 12: K-FOLD CROSS VALIDATION")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --quantization_type {QUANTIZATION_TYPE} \
    --run_full_kfold \
    --num_folds 5 \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario12_kfold

---
# SCENARIO 13: LABEL PRIORITY WEIGHTED

**Purpose:** Weight certain labels higher in evaluation

Use this when some labels are more important (e.g., detecting threats is more critical than spam)

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 13: LABEL PRIORITY WEIGHTED")
print("=" * 60)

# Priority weights: threat=3, sexual=2, others=1
LABEL_PRIORITY = '{"threat": 3, "sexual": 2, "bully": 1, "religious": 1, "spam": 1}'

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_prune_quant \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "{STUDENT_PATH}" \
    --kd_epochs {KD_EPOCHS} \
    --pruning_method {PRUNING_METHOD} \
    --target_sparsity {TARGET_SPARSITY} \
    --quantization_type {QUANTIZATION_TYPE} \
    --label_priority '{LABEL_PRIORITY}' \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario13_weighted

---
# SCENARIO 14: STUDENT MODEL COMPARISON

Compare different student architectures

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 14A: sahajBERT (Bangla Optimized)")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "neuropark/sahajBERT" \
    --kd_epochs {KD_EPOCHS} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario14a_sahajbert

In [ ]:
%%time
print("=" * 60)
print("SCENARIO 14B: DistilBERT Multilingual")
print("=" * 60)

!python main.py \
    --dataset_path {DATASET_PATH} \
    --author_name "{AUTHOR_NAME}" \
    --pipeline kd_only \
    --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
    --student_path "distilbert-base-multilingual-cased" \
    --kd_epochs {KD_EPOCHS} \
    --data_fraction {DATA_FRACTION} \
    --output_dir {RESULTS_DIR}/scenario14b_distilbert

---
# SCENARIO 15: HYPERPARAMETER GRID SEARCH

Find optimal KD hyperparameters

In [ ]:
%%time
import itertools

# Define search space
alphas = [0.5, 0.7, 0.9]
temperatures = [2.0, 4.0, 6.0]

print("=" * 60)
print("SCENARIO 15: HYPERPARAMETER GRID SEARCH")
print("=" * 60)
print(f"Total experiments: {len(alphas) * len(temperatures)}")

for alpha, temp in itertools.product(alphas, temperatures):
    exp_name = f"a{alpha}_t{temp}"
    print(f"\nRunning: alpha={alpha}, temperature={temp}")
    
    !python main.py \
        --dataset_path {DATASET_PATH} \
        --author_name "{AUTHOR_NAME}" \
        --pipeline kd_only \
        --teacher_checkpoint "{TEACHER_CHECKPOINT}" \
        --student_path "{STUDENT_PATH}" \
        --kd_alpha {alpha} \
        --kd_temperature {temp} \
        --kd_epochs 3 \
        --data_fraction 0.2 \
        --output_dir {RESULTS_DIR}/scenario15_grid/{exp_name}

---
# Results Comparison

Compare all experiment results

In [ ]:
import json
import os
import pandas as pd
from pathlib import Path

def collect_results(results_dir):
    """Collect results from all experiments."""
    results = []
    
    for root, dirs, files in os.walk(results_dir):
        for file in files:
            if file == 'final_metrics.json':
                metrics_path = os.path.join(root, file)
                scenario_name = os.path.basename(root)
                
                try:
                    with open(metrics_path) as f:
                        metrics = json.load(f)
                    
                    results.append({
                        'scenario': scenario_name,
                        'f1_macro': metrics.get('f1_macro', 0),
                        'f1_weighted': metrics.get('f1_weighted', 0),
                        'precision_macro': metrics.get('precision_macro', 0),
                        'recall_macro': metrics.get('recall_macro', 0),
                        'latency_ms': metrics.get('latency_mean_ms', 0),
                        'model_size_mb': metrics.get('model_size_mb', 0),
                        'sparsity': metrics.get('sparsity', 0),
                        'compression_ratio': metrics.get('compression_ratio', 1)
                    })
                except Exception as e:
                    print(f"Error reading {metrics_path}: {e}")
    
    return pd.DataFrame(results)

# Collect and display results
df_results = collect_results(RESULTS_DIR)

if len(df_results) > 0:
    # Sort by F1 macro
    df_results = df_results.sort_values('f1_macro', ascending=False)
    
    print("=" * 80)
    print("RESULTS COMPARISON")
    print("=" * 80)
    print(df_results.to_string(index=False))
    
    # Save to CSV
    df_results.to_csv(f'{RESULTS_DIR}/comparison.csv', index=False)
    print(f"\nResults saved to: {RESULTS_DIR}/comparison.csv")
else:
    print("No results found. Run some scenarios first!")

---
# Visualize Results

In [ ]:
import matplotlib.pyplot as plt

if len(df_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. F1 Score Comparison
    ax1 = axes[0, 0]
    df_sorted = df_results.sort_values('f1_macro', ascending=True)
    ax1.barh(df_sorted['scenario'], df_sorted['f1_macro'], color='steelblue')
    ax1.set_xlabel('F1 Macro')
    ax1.set_title('F1 Score by Scenario')
    ax1.set_xlim(0, 1)
    
    # 2. Model Size Comparison
    ax2 = axes[0, 1]
    df_sorted = df_results.sort_values('model_size_mb', ascending=True)
    ax2.barh(df_sorted['scenario'], df_sorted['model_size_mb'], color='coral')
    ax2.set_xlabel('Model Size (MB)')
    ax2.set_title('Model Size by Scenario')
    
    # 3. Latency Comparison
    ax3 = axes[1, 0]
    df_sorted = df_results.sort_values('latency_ms', ascending=True)
    ax3.barh(df_sorted['scenario'], df_sorted['latency_ms'], color='seagreen')
    ax3.set_xlabel('Latency (ms)')
    ax3.set_title('Inference Latency by Scenario')
    
    # 4. F1 vs Size Tradeoff
    ax4 = axes[1, 1]
    scatter = ax4.scatter(
        df_results['model_size_mb'], 
        df_results['f1_macro'],
        s=100, c='purple', alpha=0.7
    )
    for i, row in df_results.iterrows():
        ax4.annotate(
            row['scenario'].replace('scenario', 'S'), 
            (row['model_size_mb'], row['f1_macro']),
            fontsize=8
        )
    ax4.set_xlabel('Model Size (MB)')
    ax4.set_ylabel('F1 Macro')
    ax4.set_title('F1 vs Model Size Tradeoff')
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/comparison_plots.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlots saved to: {RESULTS_DIR}/comparison_plots.png")
else:
    print("No results to visualize.")

---
# Test Best Model with Inference

In [ ]:
# Find best model path
if len(df_results) > 0:
    best_scenario = df_results.iloc[0]['scenario']
    best_model_path = f"{RESULTS_DIR}/{best_scenario}/compressed_models/model_hf"
    
    print(f"Best model: {best_scenario}")
    print(f"Model path: {best_model_path}")
    
    # Test predictions
    test_texts = [
        "তুমি অনেক সুন্দর",  # Positive
        "তোকে মেরে ফেলব",  # Threat
        "এটা স্প্যাম মেসেজ"  # Spam
    ]
    
    for text in test_texts:
        print(f"\nText: {text}")
        !python inference.py --model_path {best_model_path} --text "{text}"
else:
    print("Run some scenarios first to test inference.")

---
# Download Results

In [ ]:
# Zip results for download
import shutil

output_zip = "compression_results"
shutil.make_archive(output_zip, 'zip', RESULTS_DIR)

print(f"Results zipped to: {output_zip}.zip")
print("Download from the Output tab on the right panel.")

---
# Summary

This notebook covered 15 scenarios:

| # | Scenario | Purpose |
|---|----------|--------|
| 1 | Baseline | Teacher performance reference |
| 2 | KD Only | Knowledge distillation |
| 3 | Prune Only | Weight pruning |
| 4 | Quant Only | Quantization |
| 5 | KD + Prune | Two-stage compression |
| 6 | KD + Quant | Distill then quantize |
| 7 | Prune + Quant | Prune then quantize |
| 8 | Full Pipeline | KD + Prune + Quant |
| 9 | Pruning Methods | Compare magnitude/gradual/wanda |
| 10 | Quant Types | Compare FP16/INT8/INT4 |
| 11 | Sparsity Levels | Compare 20%/30%/50%/70% |
| 12 | K-Fold | Robust cross-validation |
| 13 | Label Priority | Weighted evaluation |
| 14 | Student Models | Compare architectures |
| 15 | Hyperparameter Search | Find optimal settings |